# EMBER-Dyn Quickstart 05 — Dynamic Small TF-IDF + LightGBM fusion

**Dataset**: EMBER-Dyn (TD1 dynamic-only baseline + TC1 ensemble)  
**Target metrics**: TD1 ROC AUC 0.9877  ·  TC1 ensemble ROC AUC ~0.997  
**Runtime**: ~5 minutes on Colab free CPU tier

This notebook reproduces:

1. **TD1** — a dynamic-only LightGBM trained on TF-IDF of the
   Avast-CTU `behavior.summary` fields (Table 4 of the paper).
2. **TC1** — an arithmetic-mean ensemble of TD1 with the static T1
   classifier from notebook 01 (Table 5).

The TC1 result demonstrates the multi-modal-fusion paradox: the
naive late ensemble is marginally below the static-only ceiling,
establishing the architectural open challenge described in §5 of
the paper.

In [ ]:
!pip install -q pandas pyarrow scikit-learn lightgbm zstandard

## 1. Load static splits and the Small dynamic tarballs (Zenodo open tier)

In [ ]:
import io, json, re, tarfile
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import zstandard as zstd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# EMBER-Dyn open tier is archived on Zenodo (DOI in the README). Download it first
# (e.g. `pip install zenodo_get && zenodo_get <ZENODO_DOI>`) and point DATA_DIR at the
# extracted dataset folder.
# (contains static/, dynamic/small/, manifest_open.csv)
DATA_DIR = 'dataset'

# Static splits
df_train_s = pq.read_table(f'{DATA_DIR}/static/train.parquet').to_pandas()
df_val_s   = pq.read_table(f'{DATA_DIR}/static/val.parquet').to_pandas()
df_test_s  = pq.read_table(f'{DATA_DIR}/static/test.parquet').to_pandas()
feat_cols = [c for c in df_train_s.columns if c.startswith('feature_')]

# Manifest for the dynamic side (10k SHA-256 keyed)
manifest = pd.read_csv(f'{DATA_DIR}/manifest_open.csv')

# Small dynamic tarballs (41 MB total)
small_tar_paths = {split: f'{DATA_DIR}/dynamic/small/{split}.tar.zst' for split in ['train', 'val', 'test']}
print('using tarballs:', list(small_tar_paths))

## 2. Extract dynamic Small reports and tokenise behaviour fields

In [ ]:
BEH_FIELDS = ['resolved_apis', 'executed_commands', 'files',
              'read_files', 'write_files', 'delete_files',
              'read_keys', 'write_keys', 'delete_keys',
              'started_services', 'created_services', 'mutexes', 'keys']
TOK = re.compile(r'[A-Za-z0-9_\\/.\-]+')

def load_reports(tar_path):
    out = {}
    with open(tar_path, 'rb') as f:
        with zstd.ZstdDecompressor().stream_reader(f) as r:
            with tarfile.open(fileobj=r, mode='r|') as tar:
                for m in tar:
                    if not m.isfile() or not m.name.endswith('.json'):
                        continue
                    sha = m.name.rsplit('/', 1)[-1][:-5]
                    out[sha] = json.loads(tar.extractfile(m).read())
    return out

def tokens_of(report):
    summary = (report.get('behavior') or {}).get('summary') or {}
    bag = []
    for k in BEH_FIELDS:
        for s in (summary.get(k) or []):
            if isinstance(s, str):
                bag.extend(t.lower() for t in TOK.findall(s) if 2 <= len(t) <= 80)
    return ' '.join(bag)

reports = {}
for split in ['train', 'val', 'test']:
    reports.update(load_reports(small_tar_paths[split]))
print(f'loaded {len(reports):,} reports')

In [ ]:
def texts_and_labels(split):
    m = manifest[manifest['dyn_split'] == split]
    shas = m['sha256'].tolist()
    texts = [tokens_of(reports.get(s, {})) for s in shas]
    y = m['label'].astype(int).values
    return texts, y, shas

tr_texts, tr_y, tr_sha = texts_and_labels('train')
va_texts, va_y, va_sha = texts_and_labels('val')
te_texts, te_y, te_sha = texts_and_labels('test')

vec = TfidfVectorizer(max_features=50_000, min_df=5, max_df=0.9, sublinear_tf=True)
Xd_train = vec.fit_transform(tr_texts)
Xd_val   = vec.transform(va_texts)
Xd_test  = vec.transform(te_texts)
print(f'TF-IDF dim = {Xd_train.shape[1]:,}')

## 3. TD1 — dynamic-only LightGBM

In [ ]:
import lightgbm as lgb

LGB_PARAMS = dict(
    num_leaves=64, min_data_in_leaf=100, learning_rate=0.05,
    n_estimators=500, n_jobs=-1, random_state=42, verbosity=-1,
)
td1 = lgb.LGBMClassifier(objective='binary', **LGB_PARAMS)
td1.fit(Xd_train, tr_y, eval_set=[(Xd_val, va_y)],
        callbacks=[lgb.early_stopping(50, verbose=False)])
p_dyn = td1.predict_proba(Xd_test)[:, 1]
print(f'TD1  ROC AUC = {roc_auc_score(te_y, p_dyn):.4f}    '
      f'PR AUC = {average_precision_score(te_y, p_dyn):.4f}    '
      f'F1 = {f1_score(te_y, p_dyn > 0.5):.4f}')

## 4. T1 — static-only LightGBM (re-trains the model from notebook 01)

In [ ]:
def split_X_y(df):
    return df[feat_cols].values.astype('float32'), df['label'].values.astype('int32')
Xs_train, ys_train = split_X_y(df_train_s)
Xs_val,   ys_val   = split_X_y(df_val_s)
Xs_test,  ys_test  = split_X_y(df_test_s)

t0 = lgb.LGBMClassifier(objective='binary', **LGB_PARAMS)
t0.fit(Xs_train, ys_train, eval_set=[(Xs_val, ys_val)],
       callbacks=[lgb.early_stopping(50, verbose=False)])
p_stat_test = t0.predict_proba(Xs_test)[:, 1]
print(f'T1   ROC AUC = {roc_auc_score(ys_test, p_stat_test):.4f}')

## 5. TC1 — arithmetic-mean probability ensemble on the static∩dynamic test intersection

In [ ]:
static_sha = df_test_s['sha256'].tolist()
dyn_sha    = te_sha
common = sorted(set(static_sha) & set(dyn_sha))
stat_idx = {s: i for i, s in enumerate(static_sha)}
dyn_idx  = {s: i for i, s in enumerate(dyn_sha)}

p_s = np.array([p_stat_test[stat_idx[s]] for s in common])
p_d = np.array([p_dyn[dyn_idx[s]] for s in common])
y_c = np.array([te_y[dyn_idx[s]] for s in common])

p_ens = (p_s + p_d) / 2.0
print(f'TC1  ROC AUC = {roc_auc_score(y_c, p_ens):.4f}    '
      f'PR AUC = {average_precision_score(y_c, p_ens):.4f}    '
      f'F1 = {f1_score(y_c, p_ens > 0.5):.4f}    '
      f'n = {len(common):,}')

## Expected output

```
TD1  ROC AUC 0.9877    PR AUC 0.9959    F1 0.9720
T1   ROC AUC ~0.999
TC1  ROC AUC ~0.997    PR AUC ~0.999    F1 ~0.987    n ~1,907
```

TC1 (0.997) is **marginally below** T1 alone (0.999) — naive
static-plus-dynamic fusion offers no gain, which is the open
architectural challenge documented as Open Challenge 6 in §5 of
the paper.